# 04 · Detecção: threshold estático vs. dinâmico

Erro de reconstrução no teste → limiar estático (p95 do erro de treino) e dinâmico (percentil em janela móvel causal). Comparação da sensibilidade dos dois esquemas (ADR-0005).

## Setup

In [ ]:
# --- Colab ---
# !git clone https://github.com/Cerne17/NeuraTrade.git
# %cd NeuraTrade
# !pip install -e .

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import CONFIG, set_seeds
from src import data, preprocessing as pp, train as T, detect as D

set_seeds()
TICKERS = CONFIG['tickers']
pre = {t: pp.preprocess_ticker(df) for t, df in data.load_all().items()}
models = {t: T.load_model(t) for t in TICKERS}
print('modelos carregados:', list(models))

## Erro de reconstrução e limiares

Limiar estático do erro de treino; dinâmico sobre o erro de teste (janela causal).

In [ ]:
err_tr = {t: D.reconstruction_error(models[t], pre[t]['X_train']) for t in TICKERS}
err_te = {t: D.reconstruction_error(models[t], pre[t]['X_test']) for t in TICKERS}
thr_s = {t: D.static_threshold(err_tr[t]) for t in TICKERS}
thr_d = {t: D.dynamic_threshold(err_te[t]) for t in TICKERS}
# datas das janelas de teste (janela i termina em test_index[i+window-1])
W = CONFIG['preprocessing']['window_size']
dates = {t: pre[t]['test_index'][W-1:] for t in TICKERS}

## Comparação quantitativa

In [ ]:
rows = []
for t in TICKERS:
    f_s = D.flag_anomalies(err_te[t], thr_s[t])
    f_d = D.flag_anomalies(err_te[t], thr_d[t])
    rows.append({'ticker': t, 'thr_static': round(thr_s[t],4),
                 'anom_static': int(f_s.sum()), 'pct_static': round(100*f_s.mean(),1),
                 'anom_dinamico': int(f_d.sum()), 'pct_dinamico': round(100*f_d.mean(),1)})
pd.DataFrame(rows).set_index('ticker')

## Erro de teste com os dois limiares

O limiar dinâmico acompanha mudanças de regime de volatilidade; o estático é uma reta.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 8))
for ax, t in zip(axes.ravel(), TICKERS):
    d = dates[t]; e = err_te[t]
    ax.plot(d, e, lw=0.6, label='erro')
    ax.axhline(thr_s[t], color='red', ls='--', lw=1, label='estatico p95')
    ax.plot(d, thr_d[t], color='green', lw=1, label='dinamico')
    ax.set_title(t); ax.legend(fontsize=8)
fig.suptitle('Erro de reconstrucao (teste) e limiares')
fig.tight_layout(); plt.show()

## Contraste no AMER3

No período de turbulência (fraude 2023 + recuperação judicial), o limiar **estático** marca grande parte das janelas como anômalas (satura), enquanto o **dinâmico** se recalibra ao novo regime e isola melhor os eventos. É a justificativa central do esquema dinâmico.

In [ ]:
t = 'AMER3.SA'
f_s = D.flag_anomalies(err_te[t], thr_s[t])
f_d = D.flag_anomalies(err_te[t], thr_d[t])
d = dates[t]
print('janelas anomalas — estatico:', int(f_s.sum()), '| dinamico:', int(f_d.sum()))
print('\nprimeiras datas marcadas (dinamico):')
print(d[f_d][:10].date)

## Conclusões

- O limiar **estático (p95)** é simples e estável, mas, sob mudança de regime, satura (AMER3: ~27% das janelas), perdendo poder discriminativo.
- O limiar **dinâmico causal** se recalibra à volatilidade local, reduzindo marcações espúrias (AMER3: ~14%; ITUB4: 11%→5%) sem usar informação do futuro.
- Para ativos de regime estável (PETR4, VALE3) os dois esquemas quase coincidem (~3%).
- Próximo (M5): avaliação quantitativa (Precision/Recall/F1) por injeção de anomalias sintéticas, comparando os dois limiares e o baseline.